In [ ]:
import os, sys

# Make the notebook runnable both interactively (from the repo root) and via
# `jupyter nbconvert --execute` (kernel cwd = experiments/).
if not os.path.isdir(os.path.join(os.getcwd(), 'tskan')):
    _root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(_root)
    sys.path.insert(0, _root)


# Deep learning baselines (paper, Table I)

Re-runs the deep time series baselines of the paper with `tsai`:
TST (several configurations), FCN, ResNet, xresnet1d34, ResCNN, LSTM
(1-3 layers, uni/bidirectional), LSTM-FCN, InceptionTime, XceptionTime,
OmniScaleCNN and mWDN.

Protocol (paper, Section V): 70/15/15 split; the 7th variable (video height)
is dropped; 200 epochs per run; learning rates in
`[0.01, 0.001, 0.0001]` plus Adam's default; 20 restarts per configuration;
the best run (lowest train/valid RMSE) is evaluated on the held-out test split.

Set `DATASET` below to run the other dataset. **Note:** the full grid is
expensive (hundreds of 200-epoch trainings); run it in chunks or reduce
`N_RESTARTS` for a quick check.

Run from the repository root: `python -m jupyter notebook experiments/03_baselines_deep_learning.ipynb`


In [ ]:
# Configuration: pick the dataset
DATASET = 'netflix2'      # or 'waterloo3'
N_RESTARTS = 20           # restarts per (arch, lr) configuration
EPOCHS = 200


In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from tsai.all import *
from tsai.models.utils import count_parameters

# tsai's TS-format loader (tsai==0.4.0); returns (X (N, C, T) float64, y (N,))
from tsai.data.external import _ts2dfV2, _check_X

FILES = {
    'netflix2':  f'data/netflix2/netflixpluspadded_{{split}}.ts',
    'waterloo3': f'data/waterloo3/waterloo3_zscore_{{split}}.ts',
}[DATASET]


In [ ]:
Xs, ys = {}, {}
for split in ['TRAIN', 'VALID', 'TEST']:
    X, y = _ts2dfV2(FILES.format(split=split))
    Xs[split] = _check_X(X)          # (N, 7, T)
    ys[split] = y

# Live-Netflix-II keeps all 7 variables (video height is informative there);
# Waterloo 3 drops the 7th variable (video height), as in the paper.
if DATASET == 'waterloo3':
    X_train = np.delete(Xs['TRAIN'], [6], axis=1)
    X_valid = np.delete(Xs['VALID'], [6], axis=1)
    X_test  = np.delete(Xs['TEST'],  [6], axis=1)
else:
    X_train, X_valid, X_test = Xs['TRAIN'], Xs['VALID'], Xs['TEST']
y_train, y_valid, y_test = ys['TRAIN'], ys['VALID'], ys['TEST']

# Train/valid are combined for training (paper protocol)
X = concat(X_train, X_valid)
y = concat(y_train, y_valid)
splits = tsai.data.validation.get_predefined_splits(X_train, X_valid)
print('X', X.shape, '| y', y.shape, '| test', X_test.shape)


In [ ]:
tfms  = [None, [TSRegression()]]
batch_tfms = TSStandardize(by_sample=False, by_var=True)
dls = get_ts_dls(X, y, splits=splits, tfms=tfms, batch_tfms=batch_tfms)
dls.one_batch()


In [ ]:
archs = [
    (TST, {}),
    (TST, {'dropout': 0.3, 'fc_dropout': 0.3, 'n_heads': 32, 'n_layers': 3, 'd_k': 64, 'd_v': 64}),
    (TST, {'dropout': 0.4, 'fc_dropout': 0.4, 'n_heads': 32, 'n_layers': 3, 'd_k': 64, 'd_v': 64}),
    (TST, {'dropout': 0.3, 'fc_dropout': 0.3, 'n_heads': 16, 'n_layers': 3, 'd_k': 64, 'd_v': 64}),
    (TST, {'dropout': 0.4, 'fc_dropout': 0.4, 'n_heads': 16, 'n_layers': 3, 'd_k': 64, 'd_v': 64}),
    (TST, {'dropout': 0.3, 'fc_dropout': 0.3, 'n_heads': 32, 'n_layers': 3, 'd_k': 32, 'd_v': 32}),
    (TST, {'dropout': 0.4, 'fc_dropout': 0.4, 'n_heads': 32, 'n_layers': 3, 'd_k': 32, 'd_v': 32}),
    (FCN, {}), (ResNet, {}), (xresnet1d34, {}), (ResCNN, {}),
    (LSTM, {'n_layers': 1, 'bidirectional': False}), (LSTM, {'n_layers': 2, 'bidirectional': False}),
    (LSTM, {'n_layers': 3, 'bidirectional': False}), (LSTM, {'n_layers': 1, 'bidirectional': True}),
    (LSTM, {'n_layers': 2, 'bidirectional': True}),  (LSTM, {'n_layers': 3, 'bidirectional': True}),
    (LSTM_FCN, {}), (LSTM_FCN, {'shuffle': False}), (InceptionTime, {}),
    (XceptionTime, {}), (OmniScaleCNN, {}), (mWDN, {'levels': 4}),
]
losses = [0.01, 0.001, 0.0001, 0]   # 0 -> Adam default lr


In [ ]:
results = pd.DataFrame(columns=['arch', 'hyperparams', 'total params', 'valid loss',
                                'valid rmse', 'lr (0 is Adam default)', 'best restart',
                                'test rmse'])

for i, (arch, k) in enumerate(archs):
    # Guard against architectures incompatible with the installed fastai/tsai
    # versions (e.g. xresnet1d34 on fastai>=2.7.19) or that fail on a given
    # series length (e.g. mWDN on short series); they are skipped with a message.
    try:
        best_rmse, best_vals, best_iter, best_lr, best_elapsed, rmse_test = 100, None, None, None, None, None
        for lr in losses:
            for it in range(N_RESTARTS):
                model = create_model(arch, dls=dls, **k)
                if lr == 0:
                    learn = ts_learner(dls, model, metrics=rmse, opt_func=Adam)
                    lossrate = 0.001
                else:
                    learn = ts_learner(dls, model, metrics=rmse)
                    lossrate = lr

                start = time.time()
                learn.fit_one_cycle(EPOCHS, lossrate)
                elapsed = time.time() - start

                vals = learn.recorder.values[-1]              # [train_loss, valid_loss, valid_rmse]
                current_rmse = vals[2]
                if (vals[0] ** 0.5) > current_rmse:
                    current_rmse = (vals[0] ** 0.5)
                if current_rmse < best_rmse:
                    best_rmse, best_vals, best_iter, best_lr, best_elapsed = current_rmse, vals, it, lr, elapsed
                    modeldir = Path('experiments/models') / f'{arch.__name__}_{k}'
                    modeldir.mkdir(parents=True, exist_ok=True)
                    learn.export(modeldir / f'{lr}_{it}.pth')
                    _, _, preds_test = learn.get_X_preds(X_test)
                    rmse_test = float(mean_squared_error(y_test, preds_test) ** 0.5)
                    print(f'{arch.__name__} {k} lr={lr} restart={it} '
                          f'val_rmse={best_rmse:.4f} test_rmse={rmse_test:.4f} ({int(best_elapsed)}s)')

        results.loc[len(results)] = [arch.__name__, str(k), count_parameters(model),
                                     best_vals[0], best_vals[2], best_lr, best_iter, rmse_test]
        results.to_csv(f'experiments/results_dl_{DATASET}.csv', index=False)   # incremental
    except Exception as e:
        print(f'skip {arch.__name__} {k}: {type(e).__name__}: {e}')

results = results.sort_values(by='valid rmse', ascending=True, ignore_index=True)
results


In [ ]:
results.to_csv(f'experiments/results_dl_{DATASET}.csv', index=False)


**Expected results** (paper, Table I, test RMSE):

| Model | LIVE-Netflix-II | Waterloo 3 |
|---|---|---|
| TST (best configuration) | 0.2051 | 0.4274 |
| FCN | 0.2066 | 0.4152 |
| ResNet | 0.2017 | 0.4986 |
| InceptionTime | 0.2116 | 0.4904 |
| LSTM, 1-layer unidirectional | 0.1865 | 0.3889 |
| LSTM, bidirectional | 0.1850 | 0.4107 |
| LSTM-FCN | 0.2046 | 0.4329 |

Deep nets are non-deterministic across runs/hardware; expect small deviations.

Notes:

- Live-Netflix-II uses all **7** input variables (height varies across
  representations); Waterloo 3 uses **6** (height is constant). This is why
  the paper lists two parameter counts per architecture (e.g. FCN
  269,569 / 268,673).
- Parameter counts can differ by a few hundred from the paper's table
  (e.g. TST 3,350,145 vs 3,350,017) due to minor `tsai` version differences.
- Each (arch, lr) is trained 20 times with different random restarts; the
  best is selected by min(train loss sqrt, valid RMSE), then scored on test.
  Results are stochastic; expect values within a couple of percent of the
  paper.
**Verified re-runs** (this repository, 200 epochs, 4 lrs, 20 restarts; full results in
`experiments/results_dl_netflix2.csv` and `experiments/results_dl_waterloo3.csv`):

| model family | netflix2 test RMSE (paper) | waterloo3 test RMSE (paper) |
|---|---|---|
| Transformer (TST) | 0.2338 (0.2051) | 0.3859 (0.4274) |
| LSTM-FCN | 0.1854 (0.2046) | 0.4188 (0.4329) |
| FCN | 0.2025 (0.2066) | 0.4258 (0.4152) |
| ResNet | 0.1891 (0.2017) | 0.5540 (0.4986) |
| InceptionTime | 0.1985 (0.2116) | 0.4659 (0.4904) |
| LSTM (bi-dir) | 0.1854 (0.1850) | 0.3888 (0.4107) |
| LSTM (uni-dir) | 0.1985 (0.1865) | 0.3882 (0.3889) |

Notes: (i) the paper's Netflix deep-learning models use all 7 input channels, while its
Waterloo models drop channel 6, so the channel drop above is dataset-dependent; (ii) `xresnet1d34`
is skipped on fastai>=2.7.19 (API incompatibility) and `mWDN` on Waterloo (series too short for
its multi-resolution downsampling); both are skipped automatically with a message; (iii) results
are stochastic (20 restarts per learning rate) and vary run to run, as expected for this protocol.
